# Project: 멋진 챗봇 만들기 (PyTorch)

한국어 챗봇 데이터(songys/Chatbot_data)로 **Transformer 기반 챗봇**을 만드는 프로젝트입니다.

| 단계 | 내용 |
|---|---|
| Step 1 | 데이터 다운로드 (ChatbotData.csv) |
| Step 2 | 데이터 정제 (`preprocess_sentence()`) |
| Step 3 | 데이터 토큰화 (Mecab, `build_corpus()`) |
| Step 4 | Augmentation (Lexical Substitution, 데이터 3배) |
| Step 5 | 데이터 벡터화 (`<start>`/`<end>` 토큰, 단어 사전, `enc_train`/`dec_train`) |
| Step 6 | Transformer 훈련 및 예문 답변 생성 |
| Step 7 | 성능 측정 (`calculate_bleu()`) |




## 준비하기: 라이브러리 설치

Colab에는 `konlpy`가 기본 설치되어 있지 않으므로 설치합니다.
  
  

- `pip install` 로 이번 프로젝트에 필요한 5개 패키지를 설치합니다.
  - `konlpy` : 한국어 형태소 분석기 모음 (여기서 `Mecab` 클래스를 사용)
  - `python-mecab-ko` : 네이티브 Mecab이 설치돼 있지 않아도 pip만으로 동작하는 **백업용 Mecab** (KoNLPy Mecab 로드 실패 시 자동 대체)
  - `gdown` : Google Drive에 올려진 사전 훈련 Word2Vec(`ko.bin`)을 내려받기 위한 도구
  - `nltk` : Step 7에서 BLEU Score 계산에 사용
  - `gensim` : Word2Vec 모델을 로드/학습하기 위한 라이브러리
- `-q` 옵션은 설치 로그를 조용히(quiet) 출력하라는 의미입니다.

In [1]:
# ------------------------------------------------------------------
# 필요한 라이브러리 설치 (Colab 환경 기준)
# ------------------------------------------------------------------
# konlpy          : 한국어 형태소 분석기 (Mecab 클래스 사용 목적)
# python-mecab-ko : KoNLPy Mecab이 동작하지 않을 때를 대비한 백업 토크나이저
# gdown           : Google Drive 파일(ko.bin) 다운로드용
# nltk            : BLEU Score 계산용 (Step 7)
# gensim          : Word2Vec 로드/학습용 (Step 4)
!pip install -q konlpy python-mecab-ko gdown nltk gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 579.6/579.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 13.2 MB/s eta 0:00:00


## 라이브러리 버전 확인


사용할 라이브러리들을 `import` 한 뒤
각 라이브러리의 `__version__` 속성을 출력해서 **어떤 버전이 설치되어 있는지 확인**합니다.


In [2]:
# ------------------------------------------------------------------
# 사용할 라이브러리를 불러오고 버전을 확인한다
# ------------------------------------------------------------------
import numpy    # 수치 연산 (벡터화된 배열 처리)
import pandas   # CSV 데이터 로드/처리
import torch    # 딥러닝 프레임워크 (Transformer 구현)
import nltk     # 자연어 처리 도구 (BLEU Score)
import gensim   # Word2Vec (Lexical Substitution)

# 각 라이브러리의 버전 출력
print(numpy.__version__)
print(pandas.__version__)
print(torch.__version__)
print(nltk.__version__)
print(gensim.__version__)

2.0.2
2.2.2
2.11.0+cu128
3.9.1
4.4.0




실험의 **재현성**을 위해 난수 시드(seed)를 고정하고, 훈련에 사용할 장치(device)를 정합니다.

- `random`, `numpy`, `torch` 세 곳의 난수 생성기를 모두 같은 시드로 고정합니다.
  (데이터 증강의 랜덤 치환, 모델 가중치 초기화, 배치 셔플 등이 매번 같은 결과가 되도록)
- `torch.cuda.is_available()` 로 GPU 사용 가능 여부를 확인하여
  가능하면 `cuda`(GPU), 아니면 `cpu`를 사용합니다.

In [3]:
# ------------------------------------------------------------------
# 재현성을 위한 시드 고정 + 훈련 장치(device) 설정
# ------------------------------------------------------------------
import random
import numpy as np

SEED = 1234                     # 모든 난수 생성기에 사용할 시드 값
random.seed(SEED)               # 파이썬 내장 random 모듈 시드 고정 (lexical_sub 에서 사용)
np.random.seed(SEED)            # numpy 난수 시드 고정
torch.manual_seed(SEED)         # PyTorch CPU 난수 시드 고정 (가중치 초기화 등)
torch.cuda.manual_seed_all(SEED)  # PyTorch GPU 난수 시드 고정 (GPU 사용 시)

# GPU가 있으면 cuda, 없으면 cpu 를 사용
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 중인 device:", device)

사용 중인 device: cuda


## Step 1. 데이터 다운로드

[songys/Chatbot_data](https://github.com/songys/Chatbot_data) 저장소의 `ChatbotData.csv`를 다운로드합니다.



- GitHub 저장소의 raw 파일 URL을 `pandas.read_csv()` 에 직접 전달하면
  별도 다운로드 과정 없이 바로 DataFrame으로 읽어올 수 있습니다.
- 데이터는 `Q`(질문), `A`(답변), `label`(감정 분류: 0 일상, 1 부정, 2 긍정) 세 개의 열로 구성됩니다.
- `data.head()` 로 앞 5행을 출력해 데이터 모양을 눈으로 확인합니다.

In [4]:
# ------------------------------------------------------------------
# ChatbotData.csv 를 GitHub 에서 바로 읽어온다
# ------------------------------------------------------------------
import pandas as pd

# songys/Chatbot_data 저장소의 raw 파일 주소
DATA_URL = "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"

# pandas 는 URL 을 직접 받아 CSV 를 읽을 수 있다
data = pd.read_csv(DATA_URL)

print("데이터 크기:", data.shape)   # (행 개수, 열 개수) → 약 11,823 x 3
data.head()                        # 앞 5행 미리보기 (Q / A / label 열 확인)

데이터 크기: (11823, 3)


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0




읽어 온 데이터에서 질문 열(`Q`)과 답변 열(`A`)을 각각 파이썬 리스트로 변환하여
`questions`, `answers` 변수에 나눠 저장합니다. (`label` 열은 이번 프로젝트에서 사용하지 않습니다.)
두 리스트는 **같은 인덱스가 서로 짝을 이루는** 병렬(parallel) 데이터입니다.

In [5]:
# ------------------------------------------------------------------
# 질문(Q)과 답변(A)을 각각 questions, answers 리스트에 저장
# ------------------------------------------------------------------
questions = data["Q"].tolist()   # 질문 열 → 파이썬 리스트
answers = data["A"].tolist()     # 답변 열 → 파이썬 리스트 (questions[i] 의 답이 answers[i])

# 개수가 서로 같은지, 내용은 어떤지 확인
print("questions:", len(questions), "개")
print("answers  :", len(answers), "개")
print()
print("예시 질문:", questions[0])
print("예시 답변:", answers[0])

questions: 11823 개
answers  : 11823 개

예시 질문: 12시 땡!
예시 답변: 하루가 또 가네요.


## Step 2. 데이터 정제

아래 조건을 만족하는 `preprocess_sentence()` 함수를 구현합니다.

1. 영문자의 경우, **모두 소문자로 변환**합니다.
2. 영문자와 한글, 숫자, 그리고 주요 특수문자(`? . ! ,`)를 제외하곤 **정규식을 활용하여 모두 제거**합니다.

(문장부호 양옆 공백 추가 등은 우리가 사용할 토크나이저가 처리해 주므로 구현하지 않습니다.)
  

- `sentence.lower().strip()` : 영문 대문자를 소문자로 바꾸고 문장 양끝 공백을 제거합니다. (조건 1)
- `re.sub(r"[^a-z0-9ㄱ-ㅎㅏ-ㅣ가-힣?.!,]+", " ", sentence)` : 정규식의 `[^ ... ]` 는
  "괄호 안에 나열된 문자를 **제외한** 나머지"라는 뜻이므로,
  소문자 영문(`a-z`), 숫자(`0-9`), 한글 자모/완성형(`ㄱ-ㅎㅏ-ㅣ가-힣`), 주요 특수문자(`? . ! ,`)를
  **제외한 모든 문자를 공백 하나로 치환**합니다. (조건 2)
- 마지막으로 `\s+` (연속된 공백)를 공백 하나로 정리하고 양끝 공백을 다시 제거합니다.
- 마지막 두 줄의 `print` 로 특수문자·영문이 섞인 문장이 잘 정제되는지 테스트합니다.

In [6]:
# ------------------------------------------------------------------
# Step 2. 데이터 정제 함수 preprocess_sentence()
# ------------------------------------------------------------------
import re

def preprocess_sentence(sentence):
    """문장을 소문자화하고, 허용된 문자 외에는 정규식으로 모두 제거한다."""

    # [조건 1] 영문자는 모두 소문자로 변환 + 양끝 공백 제거
    #  - str() 은 혹시 숫자 등 문자열이 아닌 값이 들어와도 안전하게 처리하기 위함
    sentence = str(sentence).lower().strip()

    # [조건 2] 허용 문자(소문자 영문, 숫자, 한글, ? . ! ,)를 "제외한" 모든 문자를 공백으로 치환
    #  - [^...] : 대괄호 안 문자들을 제외한 나머지에 매칭
    #  - ㄱ-ㅎ / ㅏ-ㅣ : 한글 자음/모음 (ㅋㅋ, ㅠㅠ 같은 표현 보존)
    #  - 가-힣       : 한글 완성형 글자 전체
    sentence = re.sub(r"[^a-z0-9ㄱ-ㅎㅏ-ㅣ가-힣?.!,]+", " ", sentence)

    # 치환 과정에서 생긴 연속 공백을 하나로 정리하고, 양끝 공백 제거
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence

# 동작 확인: 특수문자(~, @, ^, ;)는 사라지고 영문은 소문자가 되어야 한다
print(preprocess_sentence("12시 땡! Hello~~ World@@ 안녕?"))
print(preprocess_sentence("SNS보면 나만 빼고 다 행복해보여^^;;"))

12시 땡! hello world 안녕?
sns보면 나만 빼고 다 행복해보여


## Step 3. 데이터 토큰화

토큰화에는 *KoNLPy*의 `Mecab` 클래스를 사용합니다.



- 먼저 `konlpy.tag.Mecab` 을 시도합니다. Mecab은 KoNLPy 분석기 중 가장 빠르고 정확한 편이지만,
  **네이티브 mecab 프로그램이 별도로 설치되어 있어야** 동작합니다.
- Colab에 네이티브 mecab이 없어서 로드에 실패하면 `except` 로 넘어가
  pip만으로 동작하는 `python-mecab-ko` 를 대신 사용합니다.
  이때 `MecabWrapper` 클래스로 감싸서 KoNLPy와 **동일한 `morphs()` 인터페이스**를 제공하므로
  이후 코드는 어느 쪽이 선택되든 똑같이 동작합니다.
- `morphs(문장)` 은 문장을 **형태소 단위 토큰의 리스트**로 잘라 줍니다.
  마지막 줄에서 예시 문장으로 토큰화 결과를 확인합니다.

In [7]:
# ------------------------------------------------------------------
# Step 3-1. 형태소 분석기(Mecab) 준비
#   KoNLPy Mecab 사용 → 실패 시 python-mecab-ko 로 자동 대체
# ------------------------------------------------------------------
try:
    # 1순위: KoNLPy 의 Mecab 클래스 (네이티브 mecab 필요)
    from konlpy.tag import Mecab
    mecab = Mecab()
    mecab.morphs("설치 확인용 문장입니다.")   # 실제로 동작하는지 한 번 호출해 확인
    print("KoNLPy Mecab 을 사용합니다.")
except Exception as err:
    # 2순위: 네이티브 mecab이 없으면 pip 만으로 동작하는 python-mecab-ko 사용
    print("KoNLPy Mecab 로드 실패 →", err)
    from mecab import MeCab

    class MecabWrapper:
        """konlpy.tag.Mecab 과 동일한 morphs() 인터페이스를 제공하는 래퍼 클래스"""
        def __init__(self):
            self._mecab = MeCab()          # python-mecab-ko 의 MeCab 객체

        def morphs(self, text):
            return self._mecab.morphs(text)  # 형태소 리스트 반환 (konlpy와 동일 형식)

    mecab = MecabWrapper()
    print("python-mecab-ko 를 사용합니다.")

# 토큰화 동작 확인: 문장이 형태소 단위 리스트로 나뉜다
print(mecab.morphs(preprocess_sentence("12시 땡! 지루하다, 놀러가고 싶어.")))

KoNLPy Mecab 로드 실패 → Install MeCab in order to use it: http://konlpy.org/en/latest/install/
python-mecab-ko 를 사용합니다.
['12', '시', '땡', '!', '지루', '하', '다', ',', '놀', '러', '가', '고', '싶', '어', '.']


아래 조건을 만족하는 `build_corpus()` 함수를 구현합니다.

1. **소스 문장 데이터**와 **타겟 문장 데이터**를 입력으로 받습니다.
2. 데이터를 앞서 정의한 `preprocess_sentence()` 함수로 정제하고, 토큰화합니다.
3. 토큰화는 **전달받은 토크나이즈 함수**를 사용합니다. (`mecab.morphs` 전달)
4. 토큰의 개수가 일정 길이 이상인 문장은 데이터에서 제외합니다.
5. **중복되는 문장은 제외**합니다. 소스는 소스대로, 타겟은 타겟대로 검사하며, 중복 쌍이 흐트러지지 않도록 쌍 단위로 제거합니다.



- `for src, tgt in zip(...)` 으로 질문·답변을 **쌍 단위로** 순회합니다.
  (쌍 단위로 처리해야 어떤 문장이 제외될 때 상대 문장도 함께 제외되어 병렬 관계가 유지됩니다.)
- 각 문장을 `preprocess_sentence()` 로 정제한 뒤, 인자로 전달받은 `tokenize_fn`(= `mecab.morphs`)으로 토큰화합니다.
- 토큰 개수가 `max_len`(기본 20) **이상**이면 그 쌍을 건너뜁니다. (조건 4)
- 중복 검사는 `seen_src`, `seen_tgt` 두 개의 `set` 으로 수행합니다.
  토큰들을 공백으로 이어붙인 문자열을 key로 만들어, **소스는 소스끼리 / 타겟은 타겟끼리** 이미 나온 문장인지 확인하고,
  어느 한쪽이라도 중복이면 **쌍 전체를 건너뛰어** 쌍이 흐트러지지 않게 합니다. (조건 5)
- 마지막에 구현한 함수로 `questions`/`answers` 를 토큰화하여 `que_corpus`, `ans_corpus` 에 저장합니다.

In [8]:
# ------------------------------------------------------------------
# Step 3-2. build_corpus(): 정제 + 토큰화 + 길이 필터 + 중복 제거
# ------------------------------------------------------------------
def build_corpus(src_data, tgt_data, tokenize_fn, max_len=20):
    """소스/타겟 문장을 정제·토큰화하고, 길이 초과·중복 문장을 쌍 단위로 제거한다.

    Args:
        src_data    : 소스 문장 리스트 (질문)          [조건 1]
        tgt_data    : 타겟 문장 리스트 (답변)          [조건 1]
        tokenize_fn : 토크나이즈 함수 (mecab.morphs)   [조건 3]
        max_len     : 이 값 이상 토큰을 가진 문장은 제외 [조건 4]

    Returns:
        (src_corpus, tgt_corpus) : 토큰 리스트들의 리스트 (서로 병렬)
    """
    src_corpus, tgt_corpus = [], []
    seen_src, seen_tgt = set(), set()   # 중복 검사용 집합 (소스용 / 타겟용 따로)

    # 질문-답변을 쌍 단위로 순회 → 제외할 때도 쌍으로 제외되어 병렬 관계 유지
    for src, tgt in zip(src_data, tgt_data):
        # [조건 2, 3] 정제 후 전달받은 함수로 토큰화
        src_tokens = tokenize_fn(preprocess_sentence(src))
        tgt_tokens = tokenize_fn(preprocess_sentence(tgt))

        # [조건 4] 토큰 개수가 max_len 이상인 문장이 있으면 그 쌍은 제외
        if len(src_tokens) >= max_len or len(tgt_tokens) >= max_len:
            continue

        # [조건 5] 중복 제거
        #  - 토큰들을 공백으로 이어붙인 문자열을 중복 검사 key 로 사용
        #  - 소스는 소스대로(seen_src), 타겟은 타겟대로(seen_tgt) 검사
        #  - 어느 한쪽이라도 이미 나온 문장이면 쌍 전체를 건너뛴다 (쌍 정렬 유지)
        src_key = " ".join(src_tokens)
        tgt_key = " ".join(tgt_tokens)
        if src_key in seen_src or tgt_key in seen_tgt:
            continue
        seen_src.add(src_key)
        seen_tgt.add(tgt_key)

        # 모든 검사를 통과한 쌍만 코퍼스에 추가
        src_corpus.append(src_tokens)
        tgt_corpus.append(tgt_tokens)

    return src_corpus, tgt_corpus


MAX_TOKEN_LEN = 20   # 토큰 개수가 이 값 이상인 문장은 제외

# questions / answers 를 각각 que_corpus / ans_corpus 로 토큰화하여 저장
que_corpus, ans_corpus = build_corpus(questions, answers, mecab.morphs,
                                      max_len=MAX_TOKEN_LEN)

print("토큰화 후 데이터 크기:", len(que_corpus), "쌍 (원본", len(questions), "쌍)")
print("질문 예시:", que_corpus[0])
print("답변 예시:", ans_corpus[0])

토큰화 후 데이터 크기: 7519 쌍 (원본 11823 쌍)
질문 예시: ['12', '시', '땡', '!']
답변 예시: ['하루', '가', '또', '가', '네요', '.']


## Step 4. Augmentation

데이터가 1만 개가량으로 적은 편이므로 **Lexical Substitution**(문장의 일부 단어를 의미가 비슷한 단어로 바꿔치기)으로 데이터를 늘립니다.

[Kyubyong/wordvectors](https://github.com/Kyubyong/wordvectors)에서 한국어로 사전 훈련된
Word2Vec 모델 **Korean (w)** 를 다운로드하여 `ko.bin` 파일을 사용합니다.



- `os.path.exists("ko.bin")` 으로 파일이 이미 있는지 확인하여 **중복 다운로드를 방지**합니다.
- 없다면 `gdown` 으로 Google Drive에서 `ko.zip`을 내려받고, `unzip` 으로 압축을 풀어 `ko.bin`을 얻습니다.
- 마지막 `ls -l ko*` 로 파일이 준비되었는지 확인합니다.


In [9]:
# ------------------------------------------------------------------
# Step 4-1. 사전 훈련된 한국어 Word2Vec (Korean (w) → ko.bin) 다운로드
# ------------------------------------------------------------------
import os

# ko.bin 이 아직 없을 때만 다운로드 (이미 있으면 건너뜀)
if not os.path.exists("ko.bin"):
    # Kyubyong/wordvectors 의 Korean (w) Google Drive 링크에서 ko.zip 다운로드
    !gdown "https://drive.google.com/uc?id=0B0ZXk88koS2KbDhXdWg1Q2RydlU" -O ko.zip
    # 압축 해제 → ko.bin 파일이 생성된다 (-o: 덮어쓰기, -q: 조용히)
    !unzip -o -q ko.zip

# ko 로 시작하는 파일 목록을 출력해 준비 상태 확인
!ls -l ko*

Downloading...
From (original): https://drive.google.com/uc?id=0B0ZXk88koS2KbDhXdWg1Q2RydlU
From (redirected): https://drive.google.com/uc?id=0B0ZXk88koS2KbDhXdWg1Q2RydlU&confirm=t&uuid=98713f94-ae62-4c3a-b00a-fb295623ed15
To: /content/ko.zip
100% 80.6M/80.6M [00:01<00:00, 77.7MB/s]
-rw------- 1 root root 50697568 Dec 21  2016 ko.bin
-rw------- 1 root root 85362829 Dec 21  2016 ko.tsv
-rw-r--r-- 1 root root 80596565 Nov 25  2019 ko.zip




- `Word2Vec.load("ko.bin")` 으로 사전 훈련된 모델을 로드하고,
  단어 벡터 부분인 `.wv`(KeyedVectors)만 꺼내 `wv` 변수에 담습니다.
- `ko.bin`은 예전 버전 gensim으로 저장된 모델이라, 설치된 gensim 버전에 따라 로드에 실패할 수 있습니다.
  실패하면 `except` 로 넘어가 **우리 챗봇 코퍼스(`que_corpus + ans_corpus`)로 Word2Vec을 직접 학습**해 대체합니다.
  (사전 훈련 모델보다 어휘는 적지만 챗봇 데이터에 나오는 단어는 오히려 잘 커버합니다.)
- 마지막으로 벡터 개수와, '남자'와 유사한 단어를 출력해 **모델이 잘 동작하는지 확인**합니다.

In [10]:
# ------------------------------------------------------------------
# Step 4-2. Word2Vec 로드 (실패 시 챗봇 코퍼스로 직접 학습해 대체)
# ------------------------------------------------------------------
from gensim.models import Word2Vec

wv = None   # 단어 벡터(KeyedVectors)를 담을 변수
try:
    # 사전 훈련된 한국어 Word2Vec(ko.bin) 로드 → 단어 벡터(.wv)만 사용
    wv = Word2Vec.load("ko.bin").wv
    print("사전 훈련된 Word2Vec(ko.bin) 로드 완료!")
except Exception as err:
    # gensim 버전 차이 등으로 로드가 실패하면 → 우리 코퍼스로 직접 학습
    print("ko.bin 로드 실패 →", err)
    print("대체: 챗봇 코퍼스로 Word2Vec 을 직접 학습합니다.")
    w2v_own = Word2Vec(
        sentences=que_corpus + ans_corpus,  # 학습 데이터: 토큰화된 질문+답변 전체
        vector_size=100,                    # 단어 벡터 차원
        window=5,                           # 주변 단어 윈도 크기
        min_count=2,                        # 2회 미만 등장 단어는 무시
        workers=4,                          # 학습 스레드 수
        epochs=30,                          # 데이터가 작으므로 여러 번 반복 학습
        seed=SEED,                          # 재현성
    )
    wv = w2v_own.wv

# 로드/학습된 단어 벡터 확인
print("단어 벡터 개수:", len(wv.index_to_key))
if "남자" in wv:   # '남자'라는 단어가 어휘에 있으면 유사 단어를 출력해 본다
    print("'남자'와 유사한 단어:", wv.most_similar("남자", topn=3))

ERROR:gensim.models.word2vec:Model load error. Was model saved using code from an older Gensim Version? Try loading older model using gensim-3.8.3, then re-saving, to restore compatibility with current code.


ko.bin 로드 실패 → 'Word2Vec' object has no attribute 'wv'
대체: 챗봇 코퍼스로 Word2Vec 을 직접 학습합니다.
단어 벡터 개수: 3497
'남자'와 유사한 단어: [('여자', 0.9234873652458191), ('로서', 0.6722632050514221), ('동성', 0.6437991261482239)]


`lexical_sub()` 함수로 문장의 일부 토큰을 Word2Vec 상 가장 유사한 단어로 치환합니다.

- **Augmentation된 `que_corpus` + 원본 `ans_corpus`** 가 병렬을 이루고,
- 반대로 **원본 `que_corpus` + Augmentation된 `ans_corpus`** 가 병렬을 이루도록 하여
- **전체 데이터가 원래의 3배**가 되도록 합니다.


- `lexical_sub(tokens, wv, p)` : 문장의 각 토큰을 순회하면서,
  그 토큰이 Word2Vec 어휘에 있고(`tok in wv`) 확률 `p`(기본 30%)에 당첨되면
  `wv.most_similar(tok, topn=1)` 이 돌려주는 **가장 유사한 단어로 치환**합니다.
  나머지 토큰은 그대로 둡니다. → 뜻이 비슷하지만 표현이 조금 다른 새 문장이 만들어집니다.
- 이 함수를 질문 전체/답변 전체에 적용해 `aug_que_corpus`, `aug_ans_corpus` 를 만듭니다.
- 마지막으로 세 묶음을 이어붙여 3배 데이터를 만듭니다.
  - 1묶음: **원본 질문 + 원본 답변**
  - 2묶음: **증강 질문 + 원본 답변**  (질문이 조금 달라져도 같은 답)
  - 3묶음: **원본 질문 + 증강 답변**  (같은 질문에 표현이 다른 답)
- `assert` 로 두 리스트 길이가 서로 같고 원본의 3배인지 검증하고, 증강 결과를 출력해 확인합니다.

In [11]:
# ------------------------------------------------------------------
# Step 4-3. Lexical Substitution 으로 데이터 3배 증강
# ------------------------------------------------------------------
def lexical_sub(tokens, wv, p=0.3):
    """각 토큰을 확률 p 로 Word2Vec 상 가장 유사한 단어로 치환한다.

    Args:
        tokens : 토큰(형태소) 리스트, 예) ["지루", "하", "다"]
        wv     : gensim KeyedVectors (단어 벡터)
        p      : 각 토큰을 치환할 확률 (0.3 = 30%)

    Returns:
        일부 토큰이 유사어로 바뀐 새 토큰 리스트
    """
    new_tokens = []
    for tok in tokens:
        # 토큰이 Word2Vec 어휘에 존재하고, 확률 p 에 당첨된 경우에만 치환
        if tok in wv and random.random() < p:
            try:
                # most_similar → [(유사단어, 유사도), ...] 에서 1등 단어를 사용
                new_tokens.append(wv.most_similar(tok, topn=1)[0][0])
            except Exception:
                # 혹시 유사어 검색이 실패하면 원래 토큰 유지
                new_tokens.append(tok)
        else:
            # 어휘에 없거나 확률에 당첨되지 않은 토큰은 그대로 유지
            new_tokens.append(tok)
    return new_tokens


# Augmentation 수행: 질문/답변 각각에 대해 증강 버전 생성
aug_que_corpus = [lexical_sub(q, wv) for q in que_corpus]  # 증강된 질문 (원본 답변과 짝)
aug_ans_corpus = [lexical_sub(a, wv) for a in ans_corpus]  # 증강된 답변 (원본 질문과 짝)

# 3배 데이터 구성 (병렬 관계가 흐트러지지 않도록 같은 순서로 이어붙임)
#   [원본 que + 원본 ans] + [증강 que + 원본 ans] + [원본 que + 증강 ans]
que_total = que_corpus + aug_que_corpus + que_corpus
ans_total = ans_corpus + ans_corpus + aug_ans_corpus

# 길이 검증: 두 리스트가 서로 같고, 원본의 정확히 3배여야 한다
assert len(que_total) == len(ans_total) == 3 * len(que_corpus)
print("원본:", len(que_corpus), "쌍 → 증강 후:", len(que_total), "쌍")
print()
print("원본 질문 :", " ".join(que_corpus[0]))
print("증강 질문 :", " ".join(aug_que_corpus[0]))
print("원본 답변 :", " ".join(ans_corpus[0]))
print("증강 답변 :", " ".join(aug_ans_corpus[0]))

원본: 7519 쌍 → 증강 후: 22557 쌍

원본 질문 : 12 시 땡 !
증강 질문 : 12 시 멋져 !
원본 답변 : 하루 가 또 가 네요 .
증강 답변 : 달째 가 또 가 네요 .


## Step 5. 데이터 벡터화

1. 타겟 데이터(`ans_total`) 전체에 `<start>` 토큰과 `<end>` 토큰을 추가합니다.
2. 챗봇은 **소스와 타겟이 같은 언어**이므로, `que_total`과 `ans_total`을 결합해
   **전체 데이터에 대한 하나의 단어 사전**을 구축하고 (→ Embedding 층 공유 가능),
   벡터화하여 `enc_train`, `dec_train`을 얻습니다.


- 특수 토큰 4종을 정의합니다.
  - `<pad>` : 문장 길이를 맞추기 위한 패딩 (인덱스 0)
  - `<unk>` : 사전에 없는 단어 (unknown)
  - `<start>` : 디코더에게 "문장 생성 시작"을 알리는 토큰
  - `<end>` : "문장 생성 끝"을 알리는 토큰
- `ans_total`이 `list` 형태이므로 프로젝트 안내의 예시(`["<start>"] + sample_data + ["<end>"]`)처럼
  리스트 덧셈만으로 모든 답변 앞뒤에 `<start>`/`<end>` 를 간단히 붙일 수 있습니다.

In [12]:
# ------------------------------------------------------------------
# Step 5-1. 타겟 데이터에 <start>, <end> 토큰 추가
# ------------------------------------------------------------------
from collections import Counter

# 특수 토큰 정의
PAD, UNK, STA, END = "<pad>", "<unk>", "<start>", "<end>"
#  <pad>   : 배치 내 문장 길이를 맞추기 위한 패딩 토큰 (인덱스 0 고정)
#  <unk>   : 단어 사전에 없는 단어를 대신하는 토큰
#  <start> : 디코더 입력의 시작을 알리는 토큰
#  <end>   : 문장 생성의 끝을 알리는 토큰

# ans_total 은 토큰 "리스트"들의 리스트이므로,
# 리스트 덧셈만으로 모든 답변의 앞뒤에 <start>/<end> 를 붙일 수 있다
ans_total = [[STA] + ans + [END] for ans in ans_total]

print("타겟 예시:", ans_total[0])   # ['<start>', ..., '<end>'] 형태 확인

타겟 예시: ['<start>', '하루', '가', '또', '가', '네요', '.', '<end>']




- `Counter` 로 `que_total + ans_total` **전체**에 등장하는 토큰의 빈도를 셉니다.
  질문과 답변을 합쳐 **하나의 단어 사전**을 만드는 이유는, 챗봇은 소스/타겟이 같은 한국어라
  인코더와 디코더가 **Embedding 층을 공유**할 수 있고 그 편이 유리하기 때문입니다.
- 사전(`vocab`)은 특수 토큰 4개를 맨 앞에 두고, 나머지 단어를 **빈도순**으로 정렬해 만듭니다.
  `<pad>` 가 반드시 인덱스 0이 되어야 이후 패딩/마스크/손실 계산에서 0을 무시하도록 처리할 수 있습니다.
- `word2idx`(단어→인덱스), `idx2word`(인덱스→단어) 두 딕셔너리를 만들어 상호 변환에 사용합니다.
- `vectorize()` 는 토큰 리스트들을 **고정 길이 정수 배열**로 바꿉니다.
  문장이 짧으면 남는 자리는 0(`<pad>`)으로 채우고, 사전에 없는 단어는 `<unk>` 인덱스로 바꿉니다.
- 인코더 입력 `enc_train` 과 디코더 입력 `dec_train` 을 만들어 크기를 출력합니다.

In [13]:
# ------------------------------------------------------------------
# Step 5-2. 단어 사전 구축 + 벡터화 → enc_train, dec_train
# ------------------------------------------------------------------
# que + ans 전체 토큰의 등장 빈도를 센다 (하나의 사전 = Embedding 공유 목적)
counter = Counter(tok for sent in (que_total + ans_total) for tok in sent)

# 단어 사전: 특수 토큰 4개를 맨 앞에 두고, 나머지는 빈도 높은 순서로 나열
#  (ans_total 에 이미 들어간 <start>/<end> 가 중복 등록되지 않도록 제외 처리)
vocab = [PAD, UNK, STA, END] + [w for w, _ in counter.most_common()
                                if w not in (PAD, UNK, STA, END)]
word2idx = {w: i for i, w in enumerate(vocab)}   # 단어 → 인덱스
idx2word = {i: w for w, i in word2idx.items()}   # 인덱스 → 단어 (생성 시 사용)
VOCAB_SIZE = len(vocab)
print("단어 사전 크기:", VOCAB_SIZE)


def vectorize(corpus, maxlen):
    """토큰 리스트들의 코퍼스를 (N, maxlen) 정수 배열로 변환한다.

    - 남는 자리는 0(<pad>) 으로 채운다 (np.zeros 로 초기화했으므로 자동)
    - 사전에 없는 단어는 <unk> 인덱스로 대체
    - maxlen 보다 긴 문장은 잘라낸다
    """
    arr = np.zeros((len(corpus), maxlen), dtype=np.int64)   # 0(<pad>) 으로 초기화
    for i, sent in enumerate(corpus):
        ids = [word2idx.get(tok, word2idx[UNK]) for tok in sent][:maxlen]
        arr[i, :len(ids)] = ids   # 앞부분만 실제 인덱스로 채움 (뒤는 패딩)
    return arr


# 가장 긴 문장에 맞춰 최대 길이 결정
ENC_MAXLEN = max(len(s) for s in que_total)   # 인코더 입력 최대 길이
DEC_MAXLEN = max(len(s) for s in ans_total)   # 디코더 입력 최대 길이 (<start>/<end> 포함)

# 벡터화 → 모델 훈련에 사용할 최종 데이터
enc_train = vectorize(que_total, ENC_MAXLEN)
dec_train = vectorize(ans_total, DEC_MAXLEN)

print("enc_train:", enc_train.shape)   # (전체 데이터 수, ENC_MAXLEN)
print("dec_train:", dec_train.shape)   # (전체 데이터 수, DEC_MAXLEN)

단어 사전 크기: 6187
enc_train: (22557, 19)
dec_train: (22557, 21)


## Step 6. 훈련하기

번역 모델에서 사용했던 **Transformer**를 PyTorch로 그대로 구현합니다.
데이터가 작으므로 과적합을 피하도록 하이퍼파라미터를 작게 튜닝했습니다.
인코더와 디코더는 **Embedding 층을 공유**합니다 (소스·타겟이 같은 언어이기 때문).

— Transformer 를 구성하는 부품들을 클래스로 정의합니다.

- `PositionalEncoding` : 어텐션은 단어 순서를 모르므로, 각 위치마다 고유한
  사인/코사인 패턴 벡터를 임베딩에 **더해서** 순서 정보를 주입합니다.
- `MultiHeadAttention` : 어텐션의 핵심 부품입니다.
  Q(질의)·K(키)·V(값)를 선형 변환으로 만든 뒤 여러 개의 head로 쪼개(`split_heads`)
  각 head가 서로 다른 관점으로 문장을 바라보게 합니다.
  `Q·Kᵀ/√d` 로 단어 간 유사도 점수를 구하고, 마스크 위치는 `-inf` 로 채워 softmax 후 0이 되게 한 다음,
  softmax 확률로 V를 가중합해 출력을 만듭니다.
- `PositionwiseFFN` : 각 위치별로 독립 적용되는 2층 완전연결망 (확장 → ReLU → 축소)입니다.
- `EncoderLayer` : Self-Attention → FFN. 각 블록마다 잔차 연결(residual) + LayerNorm + Dropout.
- `DecoderLayer` : Masked Self-Attention (미래 단어 커닝 방지) → Cross-Attention (인코더 출력 참조) → FFN.
- `Transformer` : 위 부품을 조립한 전체 모델.
  - `self.embedding` **하나**를 인코더·디코더가 공유합니다 (같은 언어의 장점).
  - `make_pad_mask` : `<pad>`(0) 위치를 True로 표시해 어텐션에서 무시하게 합니다.
  - `make_look_ahead_mask` : 상삼각 행렬로 미래 위치를 가리고, 패딩 마스크와 OR로 결합합니다.
  - `forward` : 임베딩(√d_model 스케일) → 위치 인코딩 → 인코더 N층 → 디코더 N층 → 단어 확률(logits) 출력.

In [14]:
# ------------------------------------------------------------------
# Step 6-1. Transformer 모델 정의 (PyTorch 구현)
# ------------------------------------------------------------------
import math
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    """사인/코사인 위치 인코딩.

    어텐션 연산 자체는 단어의 '순서'를 알지 못하므로,
    각 위치(pos)마다 고유한 사인/코사인 패턴을 임베딩에 더해 순서 정보를 준다.
    """
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)                    # (max_len, d_model) 버퍼
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # 위치 0..max_len-1
        # 주파수 항: 차원이 커질수록 파장이 길어진다 (논문 공식 그대로)
        div_term = torch.exp(torch.arange(0, d_model, 2).float()
                             * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)          # 짝수 차원 → sin
        pe[:, 1::2] = torch.cos(position * div_term)          # 홀수 차원 → cos
        # 학습되지 않는 상수 텐서로 등록 (모델 저장/로드 시 함께 관리됨)
        self.register_buffer("pe", pe.unsqueeze(0))           # (1, max_len, d_model)

    def forward(self, x):
        # 입력 길이만큼 잘라서 더해준다: (B, L, d_model) + (1, L, d_model)
        return x + self.pe[:, :x.size(1)]


class MultiHeadAttention(nn.Module):
    """멀티 헤드 어텐션 (Scaled Dot-Product Attention).

    여러 개의 head 가 서로 다른 관점(표현 부분공간)에서 문장을 바라보게 한다.
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model 은 n_heads 로 나누어떨어져야 합니다."
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads    # head 하나가 담당하는 차원 수

        # Q, K, V 를 만드는 선형 변환 + 최종 출력 선형 변환
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        """(B, L, d_model) → (B, n_heads, L, d_head) 로 쪼개 head 차원을 분리"""
        B, L, _ = x.size()
        return x.view(B, L, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, q, k, v, mask=None):
        # 1) 선형 변환 후 head 분리
        q = self.split_heads(self.wq(q))    # (B, H, Lq, Dh)
        k = self.split_heads(self.wk(k))    # (B, H, Lk, Dh)
        v = self.split_heads(self.wv(v))    # (B, H, Lk, Dh)

        # 2) 어텐션 점수 = Q·K^T / sqrt(d_head)  → (B, H, Lq, Lk)
        #    sqrt 로 나누는 이유: 내적 값이 너무 커져 softmax 가 포화되는 것을 방지
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)

        # 3) 마스크 적용: True 인 위치(패딩/미래 단어)는 -inf 로 채워
        #    softmax 후 확률이 0 이 되도록 한다
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        # 4) softmax 로 확률화한 뒤 V 를 가중합
        attn = F.softmax(scores, dim=-1)    # (B, H, Lq, Lk)
        out = torch.matmul(attn, v)         # (B, H, Lq, Dh)

        # 5) head 들을 다시 이어붙이고 최종 선형 변환
        B, H, L, Dh = out.size()
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.wo(out), attn


class PositionwiseFFN(nn.Module):
    """위치별 피드포워드 네트워크: 확장(d_ff) → ReLU → 축소(d_model).

    어텐션이 모은 정보를 각 위치에서 독립적으로 한 번 더 가공한다.
    """
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),   # 차원 확장
            nn.ReLU(),                  # 비선형성
            nn.Dropout(dropout),        # 과적합 방지
            nn.Linear(d_ff, d_model),   # 원래 차원으로 축소
        )

    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    """인코더 한 층: Self-Attention → FFN (각각 잔차 연결 + LayerNorm)"""
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, pad_mask):
        # 1) Self-Attention: 질문 문장 안에서 단어들끼리 서로를 참조
        attn_out, _ = self.self_attn(x, x, x, pad_mask)
        x = self.norm1(x + self.dropout(attn_out))   # 잔차 연결 + 정규화
        # 2) FFN
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


class DecoderLayer(nn.Module):
    """디코더 한 층: Masked Self-Attn → Cross-Attn(인코더 참조) → FFN"""
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)   # 마스크드 셀프 어텐션
        self.cross_attn = MultiHeadAttention(d_model, n_heads)  # 인코더-디코더 어텐션
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, look_ahead_mask, pad_mask):
        # 1) Masked Self-Attention: 아직 생성하지 않은 '미래 단어'를 보지 못하게 마스킹
        attn_out, _ = self.self_attn(x, x, x, look_ahead_mask)
        x = self.norm1(x + self.dropout(attn_out))
        # 2) Cross-Attention: Q 는 디코더에서, K/V 는 인코더 출력에서
        #    → 답변을 만들 때 질문 문장의 어느 부분을 볼지 결정
        attn_out, cross_attn = self.cross_attn(x, enc_out, enc_out, pad_mask)
        x = self.norm2(x + self.dropout(attn_out))
        # 3) FFN
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x, cross_attn


class Transformer(nn.Module):
    """인코더-디코더 Transformer 전체 모델"""
    def __init__(self, vocab_size, n_layers, d_model, n_heads, d_ff,
                 dropout=0.1, max_len=512):
        super().__init__()
        self.d_model = d_model

        # ★ 소스와 타겟이 같은 언어(한국어) → 인코더/디코더가 Embedding 층을 "공유"
        #   padding_idx=0 : <pad> 토큰의 임베딩은 0 벡터로 고정
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.dropout = nn.Dropout(dropout)

        # 인코더 층 / 디코더 층을 n_layers 개씩 쌓는다
        self.enc_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dec_layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])

        # 디코더 출력 → 단어 사전 크기의 logits (다음 단어 확률 분포)
        self.fc_out = nn.Linear(d_model, vocab_size)

    # --- 마스크 만들기 (True = 어텐션에서 가려지는 위치) ---
    def make_pad_mask(self, seq):
        """<pad>(=0) 위치를 True 로 표시. (B, L) → (B, 1, 1, L)
        가운데 1 두 개는 (head, query 길이) 방향으로 broadcast 하기 위한 차원."""
        return (seq == 0).unsqueeze(1).unsqueeze(2)

    def make_look_ahead_mask(self, seq):
        """디코더용 마스크: 패딩 마스크 + 미래 단어 마스크(상삼각 행렬)를 OR 로 결합"""
        L = seq.size(1)
        pad = self.make_pad_mask(seq)                            # (B,1,1,L)
        # triu(diagonal=1): 대각선 위쪽(=자기보다 뒤에 오는 단어)만 True 인 (L,L) 행렬
        future = torch.triu(torch.ones(L, L, dtype=torch.bool,
                                       device=seq.device), diagonal=1)
        return pad | future                                      # broadcast → (B,1,L,L)

    def embed(self, x):
        """토큰 인덱스 → 임베딩 → sqrt(d_model) 스케일링 → 위치 인코딩 → 드롭아웃"""
        out = self.embedding(x) * math.sqrt(self.d_model)
        return self.dropout(self.pos_encoding(out))

    def forward(self, enc_in, dec_in):
        # 마스크 생성
        enc_pad_mask = self.make_pad_mask(enc_in)        # 인코더: 패딩만 가림
        dec_mask = self.make_look_ahead_mask(dec_in)     # 디코더: 패딩 + 미래 단어

        # ---- Encoder: 질문 문장을 문맥 벡터로 인코딩 ----
        enc_out = self.embed(enc_in)
        for layer in self.enc_layers:
            enc_out = layer(enc_out, enc_pad_mask)

        # ---- Decoder: 인코더 출력을 참조하며 답변 표현 생성 ----
        dec_out = self.embed(dec_in)
        for layer in self.dec_layers:
            dec_out, _ = layer(dec_out, enc_out, dec_mask, enc_pad_mask)

        # 각 위치에서 "다음 단어"에 대한 logits 출력
        return self.fc_out(dec_out)   # (B, L, vocab_size)


print("Transformer 정의 완료")

Transformer 정의 완료




하이퍼파라미터를 정하고 모델 객체를 만듭니다.

- `N_LAYERS=1, D_MODEL=368, N_HEADS=8, D_FF=1024, DROPOUT=0.2` :
  데이터가 1만 쌍 정도로 작기 때문에 **층을 얕게, 드롭아웃은 높게** 잡아 과적합을 억제합니다.
  (`D_MODEL` 은 `N_HEADS` 로 나누어떨어져야 합니다: 368 ÷ 8 = 46)
- `WARMUP_STEPS=1000, BATCH_SIZE=64, EPOCHS=10` : 훈련 파라미터입니다.
- 모델을 만들어 `device`(GPU/CPU)로 옮기고, 학습 가능한 파라미터 개수를 출력해 모델 크기를 확인합니다.
- 더 좋은 답변을 얻고 싶다면 이 값들을 바꿔가며 실험해 보세요.

In [15]:
# ------------------------------------------------------------------
# Step 6-2. 하이퍼파라미터 설정 + 모델 생성
# ------------------------------------------------------------------
from torch.utils.data import TensorDataset, DataLoader

# --- 모델 하이퍼파라미터 (작은 데이터에 맞춰 튜닝 — 자유롭게 조정해 보세요!) ---
N_LAYERS = 1      # 인코더/디코더 층 수 (데이터가 작으므로 얕게)
D_MODEL  = 368    # 임베딩/은닉 차원 (N_HEADS 로 나누어떨어져야 함: 368/8=46)
N_HEADS  = 8      # 어텐션 head 수
D_FF     = 1024   # FFN 내부 확장 차원
DROPOUT  = 0.2    # 드롭아웃 비율 (과적합 방지를 위해 다소 높게)

# --- 훈련 파라미터 ---
WARMUP_STEPS = 1000   # 러닝 레이트 warmup 스텝 수
BATCH_SIZE   = 64     # 배치 크기
EPOCHS       = 10     # 전체 데이터 반복 횟수

# 모델 생성 후 GPU/CPU 로 이동
model = Transformer(VOCAB_SIZE, N_LAYERS, D_MODEL, N_HEADS, D_FF,
                    dropout=DROPOUT, max_len=512).to(device)

# 학습 가능한 파라미터 수 출력 (모델 크기 확인)
print("파라미터 수:", sum(p.numel() for p in model.parameters() if p.requires_grad))

파라미터 수: 7703115




옵티마이저·러닝 레이트 스케줄러·손실 함수·데이터 로더를 준비합니다.

- `NoamScheduler` : "Attention Is All You Need" 논문의 러닝 레이트 스케줄입니다.
  훈련 초반 `warmup_steps` 동안 러닝 레이트를 **선형으로 올렸다가**, 이후 스텝 수의 제곱근에 반비례해 **서서히 낮춥니다.**
  (공식: `lr = d_model^-0.5 × min(step^-0.5, step × warmup^-1.5)`)
  Transformer는 초반에 러닝 레이트가 크면 발산하기 쉬워 이런 warmup이 사실상 필수입니다.
- `Adam(lr=0.0, betas=(0.9, 0.98), eps=1e-9)` : 논문과 같은 설정입니다.
  초기 lr은 0으로 두고 매 스텝 스케줄러가 lr을 계산해 덮어씁니다.
- `CrossEntropyLoss(ignore_index=0)` : `<pad>`(인덱스 0) 위치는 **손실 계산에서 제외**합니다.
- `TensorDataset` + `DataLoader` : numpy 배열을 텐서로 감싸고,
  매 epoch마다 데이터를 섞어(`shuffle=True`) 배치 단위로 꺼내 줍니다.

In [16]:
# ------------------------------------------------------------------
# Step 6-3. 옵티마이저 + Noam(warmup) 스케줄러 + 손실 함수 + 데이터 로더
# ------------------------------------------------------------------
class NoamScheduler:
    """'Attention Is All You Need' 의 warmup 러닝 레이트 스케줄.

    lr = d_model^-0.5 * min(step^-0.5, step * warmup^-1.5)
      - step < warmup  : lr 이 선형으로 증가 (안정적인 초반 학습)
      - step >= warmup : lr 이 step 의 제곱근에 반비례해 감소
    """
    def __init__(self, optimizer, d_model, warmup_steps=1000):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup = warmup_steps
        self.step_num = 0                 # 지금까지 진행한 전체 스텝 수

    def step(self):
        # 1) 현재 스텝의 러닝 레이트 계산
        self.step_num += 1
        lr = (self.d_model ** -0.5) * min(self.step_num ** -0.5,
                                          self.step_num * (self.warmup ** -1.5))
        # 2) 옵티마이저의 모든 파라미터 그룹에 lr 적용
        for group in self.optimizer.param_groups:
            group["lr"] = lr
        # 3) 실제 파라미터 업데이트 수행
        self.optimizer.step()


# 옵티마이저: 논문과 동일한 Adam 설정 (lr 은 스케줄러가 매 스텝 덮어씀)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = NoamScheduler(optimizer, D_MODEL, WARMUP_STEPS)

# 손실 함수: <pad>(인덱스 0) 위치는 손실 계산에서 제외
criterion = nn.CrossEntropyLoss(ignore_index=0)

# numpy 배열 → 텐서 데이터셋 → 배치 로더 (매 epoch 셔플)
dataset = TensorDataset(torch.from_numpy(enc_train), torch.from_numpy(dec_train))
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print("배치 수 / epoch:", len(loader))

배치 수 / epoch: 353


실제 훈련 루프.

- 디코더 입력과 정답은 **한 칸씩 어긋나게(teacher forcing)** 만듭니다.
  - `dec_in = dec_batch[:, :-1]` → `<start> w1 w2 ...` (마지막 토큰 제외)
  - `dec_out = dec_batch[:, 1:]` → `w1 w2 ... <end>` (첫 토큰 제외)
  - 즉 "`<start>` 를 보면 `w1` 을, `<start> w1` 을 보면 `w2` 를 맞혀라"라고 가르치는 것입니다.
- 배치마다: 그래디언트 초기화 → 순전파(forward) → 손실 계산 → 역전파(backward)
  → 그래디언트 클리핑(폭주 방지) → 스케줄러가 lr을 갱신하며 파라미터 업데이트.
- 손실은 `(B×L, vocab)` 대 `(B×L,)` 형태로 펼쳐 계산하며, `<pad>` 위치는 자동으로 무시됩니다.
- epoch마다 평균 손실과, 패딩을 제외한 **토큰 단위 정확도**를 출력해 학습 진행을 확인합니다.

In [17]:
# ------------------------------------------------------------------
# Step 6-4. 훈련 루프 (teacher forcing)
# ------------------------------------------------------------------
from tqdm.auto import tqdm   # 진행률 표시 바

for epoch in range(1, EPOCHS + 1):
    model.train()                                  # 훈련 모드 (드롭아웃 활성화)
    total_loss, total_correct, total_count = 0.0, 0, 0

    for enc_batch, dec_batch in tqdm(loader, desc=f"Epoch {epoch:2d}", leave=False):
        # 배치를 GPU/CPU 로 이동
        enc_batch = enc_batch.to(device)
        dec_batch = dec_batch.to(device)

        # teacher forcing: 디코더 입력과 정답을 한 칸 어긋나게 구성
        dec_in = dec_batch[:, :-1]    # <start> w1 w2 ...   (모델에 주는 입력)
        dec_out = dec_batch[:, 1:]    # w1 w2 ... <end>     (모델이 맞혀야 할 정답)

        optimizer.zero_grad()                       # 이전 배치의 그래디언트 초기화
        logits = model(enc_batch, dec_in)           # 순전파 → (B, L, vocab_size)

        # 손실 계산: (B*L, vocab) vs (B*L,) 로 펼쳐서 비교 (<pad> 는 무시됨)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), dec_out.reshape(-1))
        loss.backward()                             # 역전파 (그래디언트 계산)

        # 그래디언트 클리핑: 그래디언트 폭주로 인한 발산 방지
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scheduler.step()                            # lr 갱신 + 파라미터 업데이트

        # --- 로그용 통계 집계 ---
        total_loss += loss.item() * enc_batch.size(0)
        not_pad = dec_out != 0                                  # 패딩이 아닌 위치만
        total_correct += ((logits.argmax(-1) == dec_out) & not_pad).sum().item()
        total_count += not_pad.sum().item()

    # epoch 단위 평균 손실 / 토큰 정확도 출력
    print(f"Epoch {epoch:2d} | loss {total_loss / len(dataset):.4f}"
          f" | acc {total_correct / total_count:.4f}")

Epoch  1:   0%|          | 0/353 [00:00<?, ?it/s]

Epoch  1 | loss 5.2486 | acc 0.2492


Epoch  2:   0%|          | 0/353 [00:00<?, ?it/s]

Epoch  2 | loss 3.6688 | acc 0.3522


Epoch  3:   0%|          | 0/353 [00:00<?, ?it/s]

...

Epoch  8 | loss 2.0691 | acc 0.5193


Epoch  9:   0%|          | 0/353 [00:00<?, ?it/s]

Epoch  9 | loss 1.9269 | acc 0.5419


Epoch 10:   0%|          | 0/353 [00:00<?, ?it/s]

Epoch 10 | loss 1.7929 | acc 0.5661


— 훈련된 모델로 답변을 생성합니다.

- `generate()` 함수 (greedy decoding):
  1. 질문 문장을 훈련 때와 **똑같은 과정**(정제 → Mecab 토큰화 → 인덱스 변환)으로 처리합니다.
  2. 디코더 입력을 `<start>` 하나로 시작합니다.
  3. 모델이 출력한 logits에서 **마지막 위치의 최고 확률 단어**(`argmax`)를 다음 단어로 선택하고,
     디코더 입력 뒤에 붙여 다시 예측하는 과정을 반복합니다.
  4. `<end>` 가 나오거나 최대 길이에 도달하면 멈추고, 생성된 토큰들을 문자열로 돌려줍니다.
  - `@torch.no_grad()` : 생성 시에는 그래디언트가 필요 없으므로 계산을 꺼서 메모리/속도를 아낍니다.
- 이어서 프로젝트 제출 양식에 맞춰 **예문 4개에 대한 답변(Translations)** 과
  **하이퍼파라미터 / 훈련 파라미터**를 출력합니다.

In [18]:
# ------------------------------------------------------------------
# Step 6-5. 답변 생성 (greedy decoding) + 제출 양식 출력
# ------------------------------------------------------------------
@torch.no_grad()   # 생성 시에는 그래디언트 계산 불필요 (메모리/속도 절약)
def generate(sentence, model, maxlen=DEC_MAXLEN):
    """질문 문장을 입력받아 챗봇의 답변을 생성한다 (greedy decoding)."""
    model.eval()   # 평가 모드 (드롭아웃 비활성화)

    # 1) 훈련 때와 동일한 전처리: 정제 → 토큰화 → 인덱스 변환
    tokens = mecab.morphs(preprocess_sentence(sentence))
    src_ids = [word2idx.get(tok, word2idx[UNK]) for tok in tokens]
    src = torch.tensor([src_ids], dtype=torch.long, device=device)   # (1, L)

    # 2) 디코더 입력은 <start> 토큰 하나로 시작
    dec = torch.tensor([[word2idx[STA]]], dtype=torch.long, device=device)

    # 3) 한 단어씩 생성: 마지막 위치의 argmax 를 다음 단어로 선택
    result = []
    for _ in range(maxlen):
        logits = model(src, dec)                 # (1, 현재 길이, vocab_size)
        next_id = int(logits[0, -1].argmax())    # 마지막 위치에서 최고 확률 단어
        result.append(idx2word[next_id])
        if next_id == word2idx[END]:             # <end> 가 나오면 생성 종료
            break
        # 선택한 단어를 디코더 입력 뒤에 붙이고 반복
        dec = torch.cat([dec, torch.tensor([[next_id]], dtype=torch.long,
                                           device=device)], dim=1)
    return " ".join(result)


# --- 예문에 대한 답변 생성 (제출 양식) ---
examples = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야.",
]

print("Translations")
for i, question in enumerate(examples, 1):
    print(f"> {i}. {generate(question, model)}")

print()
print("Hyperparameters")
print(f"> n_layers: {N_LAYERS}")
print(f"> d_model: {D_MODEL}")
print(f"> n_heads: {N_HEADS}")
print(f"> d_ff: {D_FF}")
print(f"> dropout: {DROPOUT}")

print()
print("Training Parameters")
print(f"> Warmup Steps: {WARMUP_STEPS}")
print(f"> Batch Size: {BATCH_SIZE}")
print(f"> Epoch At: {EPOCHS}")

Translations
> 1. 여행 을 떠나 보 세요 . <end>
> 2. 괜찮 아요 . <end>
> 3. 데이트 를 찾아보 세요 . <end>
> 4. 저 도 진짜 사랑 하 셨 군요 . <end>

Hyperparameters
> n_layers: 1
> d_model: 368
> n_heads: 8
> d_ff: 1024
> dropout: 0.2

Training Parameters
> Warmup Steps: 1000
> Batch Size: 64
> Epoch At: 10


## Step 7. 성능 측정하기

챗봇이 주어진 질문에 적절한 답변을 하는지 확인하고,
BLEU Score를 계산하는 `calculate_bleu()` 함수를 적용해 봅니다.



- `calculate_bleu(reference, candidate)` : NLTK의 `sentence_bleu` 로
  정답 답변(reference)과 모델이 생성한 답변(candidate)의 **n-gram(1~4) 겹침 정도**를 0~1 점수로 계산합니다.
  - 챗봇 답변은 짧아서 3-gram, 4-gram 이 아예 없으면 점수가 0이 되어버리므로,
    `SmoothingFunction` 으로 **smoothing** 을 적용합니다.
  - 빈 문장이 들어오면 오류 대신 0.0 을 반환하도록 방어 처리했습니다.
- 평가는 **원본(증강 전) 데이터** `que_corpus`/`ans_corpus` 에서 100개를 무작위로 뽑아 진행합니다.
  - 각 질문에 대해 `generate()` 로 답변을 만들고, `<end>` 토큰은 제거한 뒤 정답과 비교합니다.
- 마지막에 **평균 BLEU Score** 를 출력합니다.

In [19]:
# ------------------------------------------------------------------
# Step 7-1. calculate_bleu() 구현 + 샘플 100개 평균 BLEU 측정
# ------------------------------------------------------------------
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 짧은 문장에서 3/4-gram 이 없어 0점이 되는 것을 완화하는 smoothing
smoothie = SmoothingFunction().method1

def calculate_bleu(reference, candidate):
    """BLEU Score(0~1) 를 계산한다.

    Args:
        reference : 정답 답변의 토큰 리스트
        candidate : 모델이 생성한 답변의 토큰 리스트
    """
    if len(candidate) == 0:      # 빈 답변이면 0점 (sentence_bleu 오류 방지)
        return 0.0
    # weights=(0.25,)*4 : 1~4-gram 을 균등하게 반영하는 표준 BLEU-4
    return sentence_bleu([reference], candidate,
                         weights=(0.25, 0.25, 0.25, 0.25),
                         smoothing_function=smoothie)


# --- 원본(증강 전) 데이터에서 무작위 샘플을 뽑아 평균 BLEU 측정 ---
N_SAMPLES = 100
sample_indices = random.sample(range(len(que_corpus)),
                               min(N_SAMPLES, len(que_corpus)))

scores = []
for idx in tqdm(sample_indices, desc="BLEU 측정"):
    question = " ".join(que_corpus[idx])     # 토큰을 다시 문장으로 (generate 입력용)
    reference = ans_corpus[idx]              # 정답 토큰 (<start>/<end> 없음)
    # 생성 결과에서 <end> 토큰을 제거하고 토큰 리스트로 변환
    candidate = generate(question, model).replace(END, "").split()
    scores.append(calculate_bleu(reference, candidate))

print(f"\n평균 BLEU Score ({len(scores)}개 샘플): {np.mean(scores):.4f}")

BLEU 측정:   0%|          | 0/100 [00:00<?, ?it/s]


평균 BLEU Score (100개 샘플): 0.2978




BLEU 점수만으로는 답변 품질을 판단하기 어려우므로,
방금 평가한 샘플 중 5개를 골라 **질문 / 정답 답변 / 모델 답변 / BLEU 점수**를 나란히 출력해
모델이 실제로 어떤 대답을 하는지 눈으로 직접 확인합니다.

In [20]:
# ------------------------------------------------------------------
# Step 7-2. 몇 가지 샘플을 눈으로 직접 확인
# ------------------------------------------------------------------
for idx in sample_indices[:5]:
    question = " ".join(que_corpus[idx])                              # 질문
    reference = ans_corpus[idx]                                       # 정답 답변 토큰
    candidate = generate(question, model).replace(END, "").split()    # 모델 답변 토큰

    print("질문     :", question)
    print("정답 답변 :", " ".join(reference))
    print("모델 답변 :", " ".join(candidate))
    print(f"BLEU     : {calculate_bleu(reference, candidate):.4f}")
    print("-" * 60)

질문     : 올해 계속 안 좋 네
정답 답변 : 삼재 인가 봐요 .
모델 답변 : 삼재 인가 봐요 .
BLEU     : 1.0000
------------------------------------------------------------
질문     : 짝사랑 중 인 내 가 이해 가 안 돼 .
정답 답변 : 짝사랑 앞 에 장사 없 지요 .
모델 답변 : 짝사랑 앞 에 장사 없 어요 .
BLEU     : 0.6435
------------------------------------------------------------
질문     : 여친 이 오히려 섬세 하 지 못해
정답 답변 : 사람 성향 에 따라 다른 거 니 이해 해 주 세요 .
모델 답변 : 다른 사람 이 네요 .
BLEU     : 0.0174
------------------------------------------------------------
질문     : 난 천재 다
정답 답변 : 제 가 더 천재 예요 .
모델 답변 : 제 가 더 천재 예요 .
BLEU     : 1.0000
------------------------------------------------------------
질문     : 담배 너무 비 쌈
정답 답변 : 담배 피 지 마세요 .
모델 답변 : 담배 피 는 사람 만나 세요 .
BLEU     : 0.0773
------------------------------------------------------------


## 회고

- 앞서 프로젝트를 진행했던 Seq2seq 퀘스트보다 훨씬 빠르고 정확한 답변을 얻을 수 있어서 Transformer의 위력(?)을 실감할 수 있었습니다.
